# Token Classification: POS Tagging & Chunking

In [ ]:
from datasets import load_dataset

dataset = load_dataset("conll2003")



In [2]:
label_list = dataset["train"].features["ner_tags"].feature.names
print(label_list)

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [4]:
def tokenize_and_align_labels(examples):
    
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        padding="max_length",
        max_length=128,
        is_split_into_words=True
    )
    
    labels = []
    
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        
        for word_idx in word_ids:
            
            if word_idx is None:
                label_ids.append(-100)
            
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            
            else:
                label_ids.append(-100)
            
            previous_word_idx = word_idx
        
        labels.append(label_ids)
    
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)
tokenized_dataset["train"] = tokenized_dataset["train"].select(range(2000))
tokenized_dataset["validation"] = tokenized_dataset["validation"].select(range(500))

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map: 100%|██████████| 3453/3453 [00:00<00:00, 8151.29 examples/s]


In [5]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list),
    id2label={i: l for i, l in enumerate(label_list)},
    label2id={l: i for i, l in enumerate(label_list)}
)

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs",
)

In [7]:
import numpy as np
from seqeval.metrics import f1_score

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    
    true_labels = []
    true_predictions = []
    
    for pred, lab in zip(predictions, labels):
        temp_pred = []
        temp_lab = []
        
        for p, l in zip(pred, lab):
            if l != -100:
                temp_pred.append(label_list[p])
                temp_lab.append(label_list[l])
        
        true_predictions.append(temp_pred)
        true_labels.append(temp_lab)
    
    return {
        "f1": f1_score(true_labels, true_predictions)
    }

In [8]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

In [9]:
trainer.train()

100%|██████████| 500/500 [21:49<00:00,  2.45s/it]

{'loss': 0.235, 'learning_rate': 0.0, 'epoch': 2.0}


100%|██████████| 500/500 [21:54<00:00,  2.63s/it]

{'train_runtime': 1314.8967, 'train_samples_per_second': 3.042, 'train_steps_per_second': 0.38, 'train_loss': 0.23501695251464844, 'epoch': 2.0}


TrainOutput(global_step=500, training_loss=0.23501695251464844, metrics={'train_runtime': 1314.8967, 'train_samples_per_second': 3.042, 'train_steps_per_second': 0.38, 'train_loss': 0.23501695251464844, 'epoch': 2.0})

In [10]:
trainer.evaluate()

100%|██████████| 63/63 [00:45<00:00,  1.39it/s]


{'eval_loss': 0.09847302734851837,
 'eval_f1': 0.8882681564245811,
 'eval_runtime': 46.0275,
 'eval_samples_per_second': 10.863,
 'eval_steps_per_second': 1.369,
 'epoch': 2.0}

In [14]:
sentence = "Vaibhav is a Student at Trinity in Pune "

tokens = sentence.split()
inputs = tokenizer(tokens, return_tensors="pt", is_split_into_words=True)

outputs = model(**inputs)
predictions = outputs.logits.argmax(dim=2)

predicted_labels = [label_list[p.item()] for p in predictions[0]]

for token, label in zip(tokens, predicted_labels):
    print(f"{token}: {label}")

Vaibhav: O
is: B-PER
a: B-PER
Student: B-PER
at: B-PER
Trinity: O
in: O
Pune: O


I used the CoNLL-2003 dataset for token classification tasks like chunking and named entity recognition The main challenge that i faced was handling subword tokenization and aligning labels correctly with tokens. Special tokens and padding required careful handling using -100 labels here, My observations are DistilBERT performed well for token classification even on a smaller dataset. Proper preprocessing and label alignment were crucial for achieving good performance.